# Judge agreement (Cohen's kappa)

Compares the final H1 judge (Gemma 4 31B, rubric version 14) with the researcher's blind labels
on the same turns, and lists every turn where they disagree so it can be checked by hand.

- **Labels:** `data/research_annotations/judge_validation/manual_label/human_labels_<pilot>.csv`
  (exported from the `.numbers` labelling sheet; one row per turn, versions A/B/C) with
  `blind_key_<pilot>.csv` mapping each version back to U/S/P. Pilots 1-5, seed 1.
- **Judge:** `output/final/seed{SEED}/p*/h1_consistency.jsonl`, matched by `generation_run_id`,
  trial, condition, and round.
- **Kappa** is agreement beyond chance: `(p_o - p_e) / (1 - p_e)`. `NA` counts as a third
  category. Read kappa together with raw agreement: it is unstable when one label dominates.
- Left out, as in the thesis and `tools/judge_agreement_table.py`: round-1 No Contradiction
  (NA for both sides by rule) and items the researcher marked `EXCLUDE`.

Run All after changing the settings below.

In [3]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

REPO_ROOT = Path.cwd().parent  # this notebook lives in notebooks/
LABEL_DIR = REPO_ROOT / "data" / "research_annotations" / "judge_validation" / "manual_label"
LABEL_FILES = sorted(LABEL_DIR.glob("human_labels_*.csv"))
SEED = 1  # the labelled runs are seed 1 of pilots 1-5
H1_PATHS = sorted((REPO_ROOT / "output" / "final" / f"seed{SEED}").glob("p*/h1_consistency.jsonl"))
METRICS = ("context_use", "world_consistency", "no_contradiction")
EXPORT_PATH = REPO_ROOT / "output" / "comparison" / "final_judge_disagreements.csv"
[p.name for p in LABEL_FILES], [str(p.relative_to(REPO_ROOT)) for p in H1_PATHS]

(['human_labels_dumbledore_harry.csv',
  'human_labels_harry_hagrid.csv',
  'human_labels_harry_seamus.csv',
  'human_labels_luna_neville.csv',
  'human_labels_newt_mcgonagall.csv'],
 ['output/final/seed1/p01/h1_consistency.jsonl',
  'output/final/seed1/p02/h1_consistency.jsonl',
  'output/final/seed1/p03/h1_consistency.jsonl',
  'output/final/seed1/p04/h1_consistency.jsonl',
  'output/final/seed1/p05/h1_consistency.jsonl'])

## Load and join

In [4]:
def load_judge(paths):
    """Judge rows keyed by (generation_run_id, trial_id, condition, round). An incomplete
    last line (the file is still being written by a running evaluation) is skipped and counted."""
    rows, skipped = {}, 0
    for path in paths:
        with open(path, encoding="utf-8") as file:
            for line in file:
                if not line.strip():
                    continue
                try:
                    row = json.loads(line)
                except json.JSONDecodeError:
                    skipped += 1
                    continue
                rows[(row["generation_run_id"], row["trial_id"], row["context_condition"], row["round"])] = row
    return rows, skipped


def load_labels(sheet):
    """One row per turn and metric, with the blind version mapped back to its condition."""
    key = pd.read_csv(sheet.with_name(sheet.name.replace("human_labels_", "blind_key_")), keep_default_na=False)
    # keep_default_na=False keeps the label text "NA" as a label instead of a missing value
    labels = pd.read_csv(sheet, keep_default_na=False).merge(key, on=["trial_id", "version"])
    long = labels.melt(
        id_vars=["pilot_id", "generation_run_id", "trial_id", "trial_type", "context_condition", "round",
                 "player_line", "npc_reply", "note"],
        value_vars=list(METRICS), var_name="metric", value_name="label",
    )
    long["label"] = long["label"].str.strip().str.upper()
    long["round"] = long["round"].astype(int)
    return long


judge_rows, skipped_lines = load_judge(H1_PATHS)
labels = pd.concat([load_labels(sheet) for sheet in LABEL_FILES], ignore_index=True)

records = []
for _, row in labels.iterrows():
    key = (row["generation_run_id"], row["trial_id"], row["context_condition"], row["round"])
    judge = judge_rows.get(key)
    entry = judge["decisions"][row["metric"]] if judge else None
    if entry is None:
        judge_label, judge_evidence = "MISSING", None
    elif row["metric"] == "world_consistency":
        judge_label = entry["decision"]
        judge_evidence = "; ".join(
            f"{'OK' if c['consistent'] else 'CONFLICT'}: {c['claim']} (vs {c['checked_against']})"
            for c in entry.get("claims", [])
        )
    else:
        judge_label, judge_evidence = entry["decision"], entry.get("evidence")
    records.append({
        "pilot": row["pilot_id"],
        "trial": row["trial_id"].rsplit("_", 1)[-1],
        "condition": row["context_condition"],
        "round": row["round"],
        "trial_type": row["trial_type"],
        "metric": row["metric"],
        "player_line": row["player_line"],
        "npc_reply": row["npc_reply"],
        "rater": row["label"],
        "rater_reason": row["note"],
        "judge": judge_label,
        "judge_evidence": judge_evidence,
    })
joined = pd.DataFrame(records)
round_one_nc = (joined["metric"] == "no_contradiction") & (joined["round"] == 1)
scored = joined[~round_one_nc & (joined["rater"] != "EXCLUDE") & (joined["judge"] != "MISSING")]

print(f"Label rows: {len(joined)} from {len(LABEL_FILES)} sheet(s); judge rows: {len(judge_rows)}")
print(f"Incomplete judge lines skipped: {skipped_lines}")
print(f"Round-1 No Contradiction left out: {round_one_nc.sum()}   "
      f"Excluded by the rater: {(joined['rater'] == 'EXCLUDE').sum()}   "
      f"Missing judge rows: {(joined['judge'] == 'MISSING').sum()}")
print(f"Scored label pairs: {len(scored)}   researcher FAILs: {(scored['rater'] == 'FAIL').sum()}")

Label rows: 900 from 5 sheet(s); judge rows: 300
Incomplete judge lines skipped: 0
Round-1 No Contradiction left out: 165   Excluded by the rater: 0   Missing judge rows: 0
Scored label pairs: 735   researcher FAILs: 131


## Agreement and kappa

Each row shows how many label pairs it covers, raw agreement, kappa, and each side's mix
of labels. When one side used a single label for every turn in a group (for example
the rater marking every `t11` World Consistency turn PASS), chance agreement equals
observed agreement, so kappa is 0 or undefined whatever the agreement: the `note`
column flags these rows. For them, read `agreement` and the disagreement list below
instead of kappa.


In [5]:
def label_mix(counts):
    return " ".join(f"{label}:{counts[label]}" for label in ("PASS", "FAIL", "NA") if counts[label])


def kappa(frame):
    """Agreement, Cohen's kappa, each side's label mix, and a note when kappa cannot
    be read: if either side used a single label, chance agreement equals observed
    agreement and kappa is 0 (or undefined) by construction, whatever the agreement."""
    pairs = list(zip(frame["rater"], frame["judge"]))
    n = len(pairs)
    if n == 0:
        return {"n": 0, "agreement": None, "kappa": None, "rater": "", "judge": "", "note": ""}
    observed = sum(a == b for a, b in pairs) / n
    rater_counts, judge_counts = Counter(a for a, _ in pairs), Counter(b for _, b in pairs)
    expected = sum(rater_counts[l] * judge_counts[l] for l in rater_counts) / (n * n)
    value = None if expected == 1 else (observed - expected) / (1 - expected)
    note = ""
    if len(rater_counts) == 1 and rater_counts.keys() == judge_counts.keys():
        only = next(iter(rater_counts))
        note = f"both sides gave only {only}" + (": metric does not apply here" if only == "NA" else "")
    elif len(rater_counts) == 1 or len(judge_counts) == 1:
        side = "rater" if len(rater_counts) == 1 else "judge"
        note = f"{side} used one label only: kappa not meaningful, read agreement"
    return {
        "n": n,
        "agreement": round(observed, 2),
        "kappa": None if value is None else round(value, 2),
        "rater": label_mix(rater_counts),
        "judge": label_mix(judge_counts),
        "note": note,
    }


groups = {"all": scored}
for metric in ("context_use", "world_consistency", "no_contradiction"):
    groups[metric] = scored[scored["metric"] == metric]
for trial_type in sorted(scored["trial_type"].unique()):
    groups[f"type {trial_type}"] = scored[scored["trial_type"] == trial_type]
for condition in ("U", "S", "P"):
    groups[f"condition {condition}"] = scored[scored["condition"] == condition]
for pilot in sorted(scored["pilot"].unique()):
    groups[pilot] = scored[scored["pilot"] == pilot]

summary = pd.DataFrame({name: kappa(frame) for name, frame in groups.items()}).T
summary

,n,agreement,kappa,rater,judge,note
all,735,0.96,0.86,PASS:604 FAIL:131,PASS:614 FAIL:121,
context_use,300,0.94,0.84,PASS:226 FAIL:74,PASS:231 FAIL:69,
world_consistency,300,0.96,0.78,PASS:269 FAIL:31,PASS:276 FAIL:24,
no_contradiction,135,0.99,0.95,PASS:109 FAIL:26,PASS:107 FAIL:28,
type 1,180,0.96,0.52,PASS:170 FAIL:10,PASS:175 FAIL:5,
type 2,120,0.87,0.74,PASS:56 FAIL:64,PASS:66 FAIL:54,
type 3,435,0.98,0.93,PASS:378 FAIL:57,PASS:373 FAIL:62,
condition U,245,0.95,0.86,PASS:188 FAIL:57,PASS:194 FAIL:51,
condition S,245,0.94,0.82,PASS:194 FAIL:51,PASS:196 FAIL:49,
condition P,245,0.98,0.9,PASS:222 FAIL:23,PASS:224 FAIL:21,


### Kappa per metric and trial type

In [6]:
pd.DataFrame({
    (metric, f"type {t}"): kappa(scored[(scored["metric"] == metric) & (scored["trial_type"] == t)])
    for metric in ("context_use", "world_consistency", "no_contradiction")
    for t in sorted(scored["trial_type"].unique())
}).T

n agreement kappa             rater             judge                                                             note
context_use       type 1   90      0.94  0.59    PASS:82 FAIL:8    PASS:85 FAIL:5                                                                 
                  type 2   60      0.88  0.77   PASS:25 FAIL:35   PASS:30 FAIL:30                                                                 
                  type 3  150      0.97   0.9  PASS:119 FAIL:31  PASS:116 FAIL:34                                                                 
world_consistency type 1   90      0.98   0.0    PASS:88 FAIL:2           PASS:90  judge used one label only: kappa not meaningful, read agreement
                  type 2   60      0.85   0.7   PASS:31 FAIL:29   PASS:36 FAIL:24                                                                 
                  type 3  150       1.0  None          PASS:150          PASS:150                                        both sides gave only PASS
no_contradiction  type 1    0      None  None                                                                                                     
                  type 2    0      None  None                                                                                                     
                  type 3  135      0.99  0.95  PASS:109 FAIL:26  PASS:107 FAIL:28

### Confusion tables (rows = rater, columns = judge)

In [7]:
for metric in ("context_use", "world_consistency", "no_contradiction"):
    subset = scored[scored["metric"] == metric]
    print(metric)
    display(pd.crosstab(subset["rater"], subset["judge"]))

context_use


judge,FAIL,PASS
rater,,
FAIL,63,11
PASS,6,220


world_consistency


judge,FAIL,PASS
rater,,
FAIL,22,9
PASS,2,267


no_contradiction


judge,FAIL,PASS
rater,,
FAIL,26,0
PASS,2,107


## Disagreements to confirm

Every label pair where the researcher and the judge disagree, with the full dialogue, the
researcher's note, and the judge's own evidence (for World Consistency, its claim list).
The export writes these rows to a CSV with empty `researcher_verdict` and
`researcher_note` columns, so each case can be settled by hand.

In [8]:
disagree = scored[scored["rater"] != scored["judge"]].sort_values(["pilot", "trial", "condition", "round", "metric"])
print(f"{len(disagree)} disagreements")
display(disagree[["pilot", "trial", "condition", "round", "metric", "player_line", "npc_reply",
                  "rater", "rater_reason", "judge", "judge_evidence"]])

30 disagreements


,pilot,trial,condition,round,metric,player_line,npc_reply,rater,rater_reason,judge,judge_evidence
782,cb_hp_012_newt_mcgonagall,t01,U,1,world_consistency,"Remind me, Professor, where are we right now, and what time of day is it?","Mr. Scamander, I hardly see how your rather misguided attempt at humor is relevant to the chaos you’ve caused in my Great Hall. It is precisely midday, and I expect you to act with the decorum befitting a student of Hogwarts, not as if you were in some sort of circus. Now, if you would kindly refrain from further antics, I might consider allowing you to explain yourself without additional consequences.",FAIL,Able to retrieve location Great Hall but get the wrong time,PASS,"OK: Newt caused chaos in the Great Hall (vs Action Reference: Begins to scold Newt for the disruption.); OK: The location is the Great Hall (vs Scene: Location: The Great Hall, Hogwarts); OK: The time is midday (vs Scene: Time: Late Morning); OK: Newt is a student of Hogwarts (vs No supplied fact addresses this claim.)"
735,cb_hp_012_newt_mcgonagall,t06,S,1,context_use,"What do you honestly think of Nifflers, Professor?","Mr. Scamander, while I appreciate your enthusiasm for magical creatures, I must remind you that Nifflers are not an excuse for chaos in the Great Hall. You are not only disrupting your fellow students but also testing my patience beyond reasonable limits. Now, kindly explain how you intend to rectify this situation before I am forced to take further action.",PASS,,FAIL,"Kind 1: The player asks for the character's opinion on Nifflers. Response: 'while I appreciate your enthusiasm for magical creatures, I must remind you that Nifflers are not an excuse for chaos' dodges the question about her honest opinion to focus on the discipline."
798,cb_hp_012_newt_mcgonagall,t07,S,1,world_consistency,It's late evening right now. Correct me if I'm wrong.,"Mr. Scamander, while I appreciate your attempt to clarify the time of day, I must remind you that the chaos caused by your 'accident' is hardly a matter of semantics. We are not here to debate the hour; we are here to address the utter lack of responsibility displayed in allowing a Niffler to wreak havoc in the Great Hall! Now, if you would kindly refrain from further interruptions, I shall expect a proper explanation for this debacle.",FAIL,,PASS,"OK: Newt had an accident (vs No supplied fact addresses this claim.); OK: A Niffler caused havoc in the Great Hall (vs Scene: Location = The Great Hall, Hogwarts); OK: McGonagall expects an explanation for the disruption (vs Action Reference: Begins to scold Newt for the disruption.)"
799,cb_hp_012_newt_mcgonagall,t07,U,1,world_consistency,It's late evening right now. Correct me if I'm wrong.,"Mr. Scamander, while I appreciate your attempt to clarify the time of day, it does little to mitigate the chaos you’ve wrought upon my Great Hall. Your penchant for magical creatures has once again led to a most unfortunate situation, and I assure you, I will not tolerate such reckless behaviour—be it morning, noon, or night. Now, kindly explain how you intend to rectify this mess before I am forced to take further action.",FAIL,,PASS,OK: Newt caused chaos in the Great Hall (vs Action Reference: Begins to scold Newt for the disruption.); OK: Newt has a penchant for magical creatures which caused the current situation (vs No supplied fact addresses this claim.); OK: McGonagall will not tolerate reckless behavior (vs Character: known for her strict but fair nature.)
569,cb_hp_019_luna_neville,t10,S,1,context_use,You haven't practised any spells lately. Correct me if I'm wrong.,"Yeah, I know, but I’ve been thinking about that spell we practiced last week. I just want to make sure I’m ready if anything goes wrong. We can’t afford to mess this up, not with everything at stake. Just… let’s stay sharp, alright? We’ve got this together.",PASS,,FAIL,"Kind 1: The player makes a claim about a fact (Neville hasn't practiced spells). Response: 'Yeah, I kn

In [9]:
EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
disagree.assign(researcher_verdict="", researcher_note="").to_csv(EXPORT_PATH, index=False)
print("Wrote", EXPORT_PATH)

Wrote /Users/macbookproretina/Downloads/Mycomputer/Usyd/Yr4 2026/thesis/main-research/output/comparison/final_judge_disagreements.csv
